# What ELIZA Can Do So Far - Sample Solution

COSC 1210 (CS I) Fall 26 | Augustana U.

**Name:** Birkenkrahe, Marcus

Four functions from the ELIZA lectures. For each one: write the body
under the signature, run the test cell, then answer each question in
the Answer cell below it (double-click to edit).

## 1. Spot a keyword

Write `keywords.txt` first: one keyword per line.

In [1]:
open("keywords.txt","w").write("mother\nfather\nbrother\nsister\ndream")
print(open("keywords.txt").read())

mother
father
brother
sister
dream


Print every keyword that is in the sentence:
`find_keyword("I dreamt about my mother.")` prints `mother` and `dream`.

In [2]:
def find_keyword(sentence):
    """Print every keyword from keywords.txt that is in sentence."""
    for line in open("keywords.txt"):
        word = line.strip()
        if word in sentence.lower():
            print(word)
    return None

In [3]:
find_keyword("I dreamt about my mother.")

mother
dream


1. Why do you need `.strip()` for every line of the file?

**Answer:** Each line ends with a newline. Without `strip()`, the function looks for `"mother\n"`, which is not in the sentence.

2. Why must `return None` be aligned with `for`, and not with `if`?

**Answer:** Aligned with `for`, it runs after the loop has checked every line. Aligned with `if`, it is inside the loop and returns after the first line: only `mother` is checked.

## 2. The first reply

Change `find_keyword` so that it returns the first keyword it finds, or
`None`, and give it a doctest (question 1). Then write `respond`: for
`mother` it returns "Tell me more about your mother.", for `father`
"How do you get along with your father?", for any other sentence
"Please go on."

In [4]:
def find_keyword(sentence):
    """Return the first keyword from keywords.txt in sentence, or None.

    >>> find_keyword("My sister calls my mother.")
    'mother'
    """
    for line in open("keywords.txt"):
        word = line.strip()
        if word in sentence.lower():
            return word
    return None

def respond(sentence):
    """Return ELIZA's reply to sentence."""
    keyword = find_keyword(sentence)
    if keyword == "mother":
        return "Tell me more about your mother."
    elif keyword == "father":
        return "How do you get along with your father?"
    else:
        return "Please go on."

In [5]:
from doctest import run_docstring_examples
run_docstring_examples(find_keyword, globals())
print(respond("My sister calls my mother."))
print(respond("My grandfather is 90."))
print(respond("I miss my brother."))

Tell me more about your mother.
How do you get along with your father?
Please go on.


1. Write the docstring test (doctest) for `find_keyword` with the sentence "My sister calls my mother." (in the code cell above).

**Answer:** the two lines `>>> find_keyword("My sister calls my mother.")` and `'mother'` in the docstring above. `run_docstring_examples` prints nothing when the test passes.

2. Why does "My sister calls my mother." pick `mother`?

**Answer:** The loop goes through the file, not the sentence: `mother` comes before `sister` in the file, so it is found first.

3. Why does "My grandfather is 90." pick `father`?

**Answer:** `in` finds a substring, not a word: "grandfather" contains "father".

4. Why does "I miss my brother." not get a question about the brother?

**Answer:** `brother` is a keyword, but `respond` has no branch for it, so the `else` gives the fallback "Please go on."

5. What breaks if `return None` moves under the `if`?

**Answer:** The function returns after the first line of the file, so only `mother` is checked: "My grandfather is 90." gets `None` and the reply "Please go on."

## 3. Why are you tired?

Reuse the end of the sentence in the reply: "I'm SO hungry!!" gets "Why
are you so hungry?", and a sentence that does not start with "I am"
gets `None`. Write `normalize` first (question 1): it returns the
sentence in lowercase, with "i'm" written out as "i am", without `.!?`
at the end.

In [6]:
def normalize(sentence):
    """Return sentence in lowercase, "i'm" written out, no .!? at the end."""
    return sentence.lower().replace("i'm", "i am").strip(".!?")

def i_am_reply(sentence):
    """Return "Why are you X?" for a sentence "I am X", else None."""
    sentence = normalize(sentence)
    if sentence.startswith("i am "):
        return f"Why are you {sentence[5:]}?"
    return None

In [7]:
print(i_am_reply("I'm SO hungry!!"))
print(i_am_reply("You are hungry."))

Why are you so hungry?
None


1. Write the body of `normalize` in one line (in the code cell above).

**Answer:** `return sentence.lower().replace("i'm", "i am").strip(".!?")` - see the code cell above.

2. Which characters of `sentence` does `sentence[5:]` contain?

**Answer:** From index 5 to the end. Indexes 0 to 4, "i am ", are left out: "i am hungry" gives "hungry".

3. What does `i_am_reply("You are hungry.")` return, and why?

**Answer:** `None`. After `normalize` the sentence is "you are hungry", which does not start with "i am ", so the function skips the `if` and returns `None`.

## 4. Why do you need a break?

Cut the words after "I need" out of the sentence with a pattern and a
group: "Honestly, I need my brother." gets "Why do you need my
brother?", and a sentence without "I need" gets `None`. Use
`re.search` and `group(1)`.

In [8]:
import re

def need_reply(sentence):
    """Return "Why do you need X?" for a sentence with "I need X", else None."""
    match = re.search("i need (.*)", normalize(sentence))
    if match is None:
        return None
    return f"Why do you need {match.group(1)}?"

In [9]:
print(need_reply("Honestly, I need my brother."))
print(need_reply("I need a break and I need sleep."))
print(need_reply("I need."))

Why do you need my brother?
Why do you need a break and i need sleep?
None


1. What does `.*` in `"i need (.*)"` match?

**Answer:** `.` is any character, `*` repeats it zero or more times, as many as possible: everything after "i need " up to the end of the sentence.

2. What does `need_reply("I need a break and I need sleep.")` return?

**Answer:** "Why do you need a break and i need sleep?" - `.*` takes the second "i need" too.

3. Why does `need_reply("I need.")` return `None`?

**Answer:** After `normalize` it is "i need": there is no space after "need", and the pattern needs one before `(.*)`.

4. What is the difference between `match.group()` and `match.group(1)`?

**Answer:** `group()` is the whole match, "i need my brother"; `group(1)` is only the part in the first pair of parentheses, "my brother".

5. Why does `r"\b(mother|father|sister|brother)\b"` start with `r`?

**Answer:** Without `r`, Python turns `\b` into one character, a backspace, before `re` sees the pattern, and no sentence contains a backspace. With `r`, `re` gets `\b`, a word boundary.